# Research Paper Implementation

*Auto-generated from PDF using AI pipeline*

---

## 📄 Paper Summary

- **Summary of the Academic Paper: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding**
- **Main Objective / Problem Being Solved:**
- The main objective of this paper is to introduce a new language representation model called BERT (Bidirectional Encoder Representations from Transformers) that can be pre-trained on unlabeled text data and fine-tuned for a wide range of natural language processing (NLP) tasks. The problem being solved is the limitation of existing language models, which are unidirectional and restrict the power of pre-trained representations, especially for fine-tuning approaches.
- **Key Methodology or Approach:**
- The key methodology used in this paper is the introduction of a "masked language model" (MLM) pre-training objective, which randomly masks some tokens from the input and predicts the original vocabulary id of the masked word based on its context. This approach enables the representation to fuse left and right context, allowing for pre-training of deep bidirectional Transformers. Additionally, a "next sentence prediction" task is used to jointly pre-train text-pair representations.
- **Main Findings or Results:**
- The main findings of this paper are:
- * BERT achieves state-of-the-art results on eleven NLP tasks, including question answering, language inference, and named entity recognition.
- * BERT obtains new state-of-the-art results on several benchmark datasets, including GLUE, MultiNLI, and SQuAD.
- * The pre-trained BERT model can be fine-tuned with just one additional output layer to create state-of-the-art models for a wide range of tasks.
- **Why This Matters:**
- This paper matters because it introduces a new and powerful language representation model that can be pre-trained on unlabeled text data and fine-tuned for a wide range of NLP tasks. BERT's ability to learn deep bidirectional representations from unlabeled text data has the potential to improve the performance of many NLP tasks, and its simplicity and flexibility make it a valuable tool for the NLP community. The findings of this paper also demonstrate the importance of bidirectional pre-training for language representations and show that pre-trained representations can reduce the need for heavily-engineered task-specific architectures.

## 🔢 Key Mathematical Equations

**Equation 1**

$$
BERTBASE (L=12, H=768, A=12, Total Param-
$$

---

**Equation 2**

$$
eters=110M) and BERTLARGE (L=24, H=1024,
$$

---

**Equation 3**

$$
A=16, Total Parameters=340M).
$$

---

**Equation 4**

$$
i.e., 3072 for the H = 768 and 4096 for the H = 1024.
$$

---

**Equation 5**

$$
all of the words in the paragraph: Pi =
$$

---

**Equation 6**

$$
compare the score of the no-answer span: snull =
$$

---

**Equation 7**

$$
si,j = maxj≥iS·Ti + E·Tj. We predict a non-null
$$

---

**Equation 8**

$$
Vaswani et al. (2017) is (L=6, H=1024, A=16)
$$

---

**Equation 9**

$$
is (L=64, H=512, A=2) with 235M parameters
$$

---

**Equation 10**

$$
Ablation over BERT model size. #L = the
$$

---



## 📦 Requirements

```bash
pip install numpy scipy matplotlib
```

## 💻 Implementation

### Step 1

In [ ]:
import torch

### Step 2

In [ ]:
import torch.nn as nn

### Step 3

In [ ]:
import torch.optim as optim

### Step 4

In [ ]:
import numpy as np

### Step 5

In [ ]:
import math

### Step 6

In [ ]:
class MaskedLanguageModel(nn.Module):
    def __init__(self, hidden_size, vocab_size, dropout=0.1):
        super(MaskedLanguageModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.transformer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=8, dim_feedforward=hidden_size, dropout=dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output = self.transformer(embedded)
        output = torch.mean(output, dim=1)
        output = self.fc(output)
        return output

### Step 7

In [ ]:
class BERT(nn.Module):
    def __init__(self, hidden_size, vocab_size, num_layers=12, dropout=0.1):
        super(BERT, self).__init__()
        selfmasked_language_model = MaskedLanguageModel(hidden_size, vocab_size, dropout=dropout)
        self.transformer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=8, dim_feedforward=hidden_size, dropout=dropout)
        self.fc1 = nn.Linear(hidden_size, 768)
        self.dropout = nn.Dropout(p=dropout)
        self.fc2 = nn.Linear(768, vocab_size)

    def forward(self, x):
        output = self.masked_language_model(x)
        output = self.transformer(output)
        output = self.dropout(output)
        output = self.fc1(output)
        output = torch.relu(output)
        output = self.fc2(output)
        return output

### Step 8

In [ ]:
class BERTBase(BERT):
    def __init__(self, vocab_size):
        super(BERTBase, self).__init__(hidden_size=768, vocab_size=vocab_size, num_layers=12, dropout=0.1)

### Step 9

In [ ]:
class BERTLarge(BERT):
    def __init__(self, vocab_size):
        super(BERTLarge, self).__init__(hidden_size=1024, vocab_size=vocab_size, num_layers=24, dropout=0.1)

### Step 10

In [ ]:
class BERTModel:
    def __init__(self, model_name, vocab_size, mask_prob=0.8, rand_prob=0.1, same_prob=0.1):
        self.model_name = model_name
        self.model = self.get_model(model_name, vocab_size)
        self.mask_prob = mask_prob
        self.rand_prob = rand_prob
        self.same_prob = same_prob

    def get_model(self, model_name, vocab_size):
        if model_name == 'BERTBase':
            return BERTBase(vocab_size)
        elif model_name == 'BERTLarge':
            return BERTLarge(vocab_size)
        else:
            raise ValueError('Invalid model name')

    def train(self, x, y):
        input_ids = torch.randint(high=self.model.vocab_size, size=(x.shape[0], x.shape[1]))
        attention_mask = torch.randint(high=2, size=(x.shape[0], x.shape[1]), dtype=torch.bool)
        output = self.model(input_ids)
        loss_fn = nn.CrossEntropyLoss()
        loss = loss_fn(output, y)
        optimizer = optim.Adam(params=self.model.parameters(), lr=1e-4)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        return loss.item()

    def ablate(self, x, y, mask_strategy, ablation_set):
        if mask_strategy == 'MASK':
            input_ids = x.clone()
            input_ids[input_ids == 0] = 1
        elif mask_strategy == 'SAME':
            input_ids = x.clone()
        elif mask_strategy == 'RND':
            input_ids = torch.randint(high=self.model.vocab_size, size=x.shape)
        else:
            raise ValueError('Invalid masking strategy')
        output = self.model(input_ids)
        loss_fn = nn.CrossEntropyLoss()
        loss = loss_fn(output, y)
        return loss.item()

### Step 11

In [ ]:
def get_masked_input(x, vocab_size, mask_prob, rand_prob, same_prob):
    rand_idx = torch.randint(high=x.shape[1], size=(x.shape[0],), dtype=torch.long)
    same_idx = torch.randint(high=x.shape[1], size=(x.shape[0],), dtype=torch.long)
    mask_idx = torch.randint(high=x.shape[1], size=(x.shape[0],), dtype=torch.long)
    for i in range(x.shape[0]):
        if np.random.rand() < mask_prob:
            x[i, mask_idx[i]] = 1
        elif np.random.rand() < rand_prob:
            x[i, rand_idx[i]] = torch.randint(high=vocab_size, size=(1,))
        elif np.random.rand() < same_prob:
            x[i, same_idx[i]] = x[i, same_idx[i]]
    return x

vocab_size = 10000
num_samples = 10
batch_size = 32
model_name = 'BERTBase'

model = BERTModel(model_name, vocab_size)
x = torch.randint(high=vocab_size, size=(num_samples, batch_size))
y = torch.randint(high=vocab_size, size=(num_samples, batch_size))
for _ in range(10):
    loss = model.train(x, y)
    print(f'Loss: {loss}')

x = get_masked_input(x, vocab_size, mask_prob=0.8, rand_prob=0.1, same_prob=0.1)
y = torch.randint(high=vocab_size, size=(num_samples, batch_size))
for mask_strategy in ['MASK', 'SAME', 'RND']:
    for ablation_set in [0.8, 0.9]:
        loss = model.ablate(x, y, mask_strategy, ablation_set)
        print(f'Mask Strategy: {mask_strategy}, Ablation Set: {ablation_set}, Loss: {loss}')